In [1]:
class ExtendedEnv:
    def __init__(self):
        self.state = 0
        self.end_state = 5
        self.states = list(range(0, self.end_state + 1))  # tous les états
        self.actions = [0, 1]  # 0 = reculer, 1 = avancer

    def reset(self):
        self.state = 0
        return self.state

    def step(self, action):
        if action == 1 and self.state < self.end_state:
            self.state += 1
        elif action == 0 and self.state > 0:
            self.state -= 1

        reward = 1 if self.state == self.end_state else 0
        done = (self.state == self.end_state)
        return self.state, reward, done, {}

    def get_all_states(self):
        return self.states

    def get_all_actions(self):
        return self.actions


In [2]:
import random

def generate_episode(env, policy):
    episode = []
    state = env.reset()
    done = False

    while not done:
        actions = list(policy[state].keys())
        probs = list(policy[state].values())
        action = random.choices(actions, weights=probs, k=1)[0]  # TODO understand
        next_state, reward, done, _ = env.step(action)
        episode.append((state, action, reward))
        state = next_state

    return episode

In [3]:
def off_policy_mc_control(env, num_episodes):
    all_states = env.get_all_states()
    all_actions = env.get_all_actions()
    gamma = 0.99

    target_policy = {}
    for state in all_states:
        target_policy[state] = all_actions[0]

    behavior_policy = {}
    for state in all_states:
        behavior_policy[state] = {}
        for action in all_actions:
            behavior_policy[state][action] = 0.1

    Q = {}
    for state in all_states:
        Q[state] = {}
        for action in all_actions:
            Q[state][action] = 0.0

    # 🔹 Cumul des poids pour importance sampling
    C = {}
    for state in all_states:
        C[state] = {}
        for action in all_actions:
            C[state][action] = 0.0


    for episode_num in range(num_episodes):
        episode = generate_episode(env, behavior_policy)

        G = 0.0
        W = 1.0

        for i in reversed(range(len(episode))):
            state, action, reward = episode[i]
            G = gamma * G + reward

            # 🔹 Update pondéré de Q avec importance sampling
            C[state][action] += W
            Q[state][action] += (W / C[state][action]) * (G - Q[state][action])

            # 🔹 Update de la target policy (greedy sur Q)
            best_action = max(Q[state], key=Q[state].get)
            target_policy[state] = best_action

            # 🔹 Si l’action diffère de l’action target (greedy), stoppe l’update
            if action != best_action:
                break

            # 🔹 Importance sampling: pondération
            prob_b = behavior_policy[state][action]
            if prob_b == 0:
                break
            W = W / prob_b

    return target_policy, Q


In [4]:
env = ExtendedEnv()
print(off_policy_mc_control(env, 10000))

({0: 1, 1: 1, 2: 1, 3: 1, 4: 1, 5: 0}, {0: {0: 0.9509900498999999, 1: 0.96059601}, 1: {0: 0.9509900498999999, 1: 0.9702989999999999}, 2: {0: 0.96059601, 1: 0.9801}, 3: {0: 0.9702989999999999, 1: 0.99}, 4: {0: 0.9801, 1: 1.0}, 5: {0: 0.0, 1: 0.0}})
